### This notebook collects links and information from a given website url and generates a business brochure. 

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import gradio as gr 



/home/dave/Documents/Courses/aiengr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#only needed in Jupyter notebook
!uv pip install nest_asyncio

Using Python 3.13.18 environment at: /home/dave/Documents/Courses/aiengr
Checked 1 package in 9ms


In [ ]:
# Only needed in Jupyter
#import nest_asyncio
#nest_asyncio.apply()

In [3]:
#Initialize LLM and load environment variables

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

#verify the key

if not api_key:
    print("No API key found")
elif not api_key.startswith("sk-proj-"):
    print("Invalid API key format")
elif api_key.strip() != api_key:
    print("An API key should not have leading or trailing whitespace")
else:
    print("API key is valid")

# name the model to use (note: this should be a variable in a future edition)
MODEL = 'gpt-6-luna'
openai = OpenAI()




API key is valid


The next cells will complete the following steps:
1. structure a system prompt as a constant
2. create a user promption function to allow changing of the entered URL to be used as the basis of the brochure
3. create a function to select the relevant links from the URL given by user input in step 2
4. Create a function to stich together the page contents and relevant links (fetch_page_and_all_relevant_links)
5. Create a brochure system prompt to instuct the LLM how to create the brochure
6. Create a brochure user prompt function for the user to enter a company name and URL
7. Create a function to generate the brochure (create_brochure)
8. Add gradio



In [4]:
# 1. Structure a system prompt as a constant

link_system_prompt = """ 
You are provided with a list of links found on a webpage. 
You are able to decide which of the links would be most relevant to include in a brochure about the company, 
such as link to an About page, a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about},
        {"type": "careers page", "url": "https://full.url/goes/here/careers"}
    ]
}
"""

In [5]:
# 2  create a user promption function to allow changing of the entered URL to be used as the basis of the brochure

def get_links_user_prompt(url):
    user_prompt = f""" 
    Here is a list of links on the website {url} - 
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https url in JSON format. 
    Do not include Terms of Service or Privacy Policy email links. 

    """

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))

 
    Here is a list of links on the website https://edwarddonner.com - 
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https url in JSON format. 
    Do not include Terms of Service or Privacy Policy email links. 

    


In [7]:
# 3. create a function to select the relevant links from the URL given by user input in step 2

def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        reasoning_effort="none",
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
    response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links



In [9]:
# Test cell for select_relevant_links function

select_relevant_links("https://edwarddonner.com")

{'links': []}

In [9]:
# 4. Create a function to stich together the page contents and relevant links (fetch_page_and_all_relevant_links)

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url, use_browser=False)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{contents}\n##Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        try:
            result += fetch_website_contents(link["url"], use_browser=False)
        except Exception as e:
            result += f"(Could not fetch this page: {e})"
    return result


In [11]:
# Test cell for fetch_page_and_all_relevant_links function

print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

HTTPError: 429 Client Error: Too Many Requests for url: https://edwarddonner.com/

In [10]:
# 5. Create a brochure system prompt to instuct the LLM how to create the brochure

brochure_system_prompt = """ 
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [12]:
# 6. Create a brochure user prompt function for the user to enter a company name and URL

def get_brochure_user_prompt(company_name, url):
    user_prompt = f""" 
You are looking at a company called: {company_name}
Here are teh contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n 
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [11]:
# Test cell for get_brochure_user_prompt function
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

' \nYou are looking at a company called: HuggingFace\nHere are teh contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n \n## Landing Page: \n\nHugging Face – The AI community building the future.\n\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nUpdated\n3 days ago\n•\n14.4k\n•\n1.24k\nUpdated\n12 days ago\n•\n26.5k\n•\n1.89k\nUpdated\n6 days ago\n•\n30.7k\n•\n1.88k\nUpdated\nabout 5 hours ago\n•\n667\n•\n480\nUpdated\nabout 20 hours ago\n•\n6.48k\n•\n490\nBrowse 2M+ models\nSpaces\nThe Ultimate MoE Training Handbook\n🧩\n537\nA hands-on guide to training Mixture-of-Experts models\nImage to Video\n🎬\n756\nMake videos from text or photos — free, no GPU queue\nHarness Leaderboard\n🧭\n274\nWhich coding-agent harness should yo

In [13]:
# 7. Create a function to generate the brochure (create_brochure)

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none", 
        messages = [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    return response.choices[0].message.content
   


In [38]:
#test cell for create_brochure

create_brochure("cbs", "https://www.cbsnews.com")

# CBS News

**CBS News** delivers breaking news, top stories and exclusive coverage. Its landing page highlights political reporting, including coverage of President Trump’s *60 Minutes* interview, with key takeaways and a full transcript.

## What CBS News offers
- News coverage and reporting
- Interviews and exclusive content
- A team of anchors, correspondents and contributors

## People and leadership
CBS News lists **Tom Cibrowski** as President and Executive Editor. **David Agus, M.D.** is featured as a contributor.

## Advertising and brand partnerships
**CBS News Brand Studio** provides content marketing and advertising services. The newsroom is not involved in creating Brand Studio content.

*Note: The available source pages did not provide verifiable details about company culture, customers, or current career opportunities.*

In [15]:
# 8. Add Gradio

company_input = gr.Textbox(label="Company Name", info = "Enter the company name")
url_input = gr.Textbox(label="Input site", info="Enter a URL")
brochure_output = gr.Markdown(label="Brochure")

view = gr.Interface(
    fn= create_brochure,
    title="Create Brochure",
    inputs =[company_input, url_input],
    outputs=[brochure_output],
    flagging_mode="never"
)

view.launch(inbrowser=True)



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Gtk-Message: 14:43:59.956: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
Traceback (most recent call last):
  File "/home/dave/Documents/Courses/aiengr/lib/python3.13/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/home/dave/Documents/Courses/aiengr/lib/python3.13/site-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/home/dave/Documents/Courses/aiengr/lib/python3.13/site-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/home/dave/Documents/Courses/aiengr/lib/python3.13/site-packages/gradio/blocks.py